# N4 — Path-Tracking Control: Pure Pursuit on a Planned Path

This notebook closes the **planning-to-action loop** from Workshop 4.2 by driving a kinematic vehicle along a planned path using classical geometric controllers. We implement the kinematic bicycle model, build a pure-pursuit controller from first principles, sweep the lookahead distance to understand the aggression/stability/accuracy tradeoff, and compare against the Stanley controller.

**What you will learn:**
1. Apply the kinematic bicycle model as a tractable approximation of vehicle dynamics
2. Implement a geometric path-tracking controller (pure pursuit) from first principles
3. Tune the lookahead distance and reason about the aggression/stability/accuracy tradeoff
4. Connect path planning (Workshop 4.2) to path execution (this notebook)
5. Place pure pursuit and Stanley on the broader control spectrum: classical → optimal (LQR, MPC) → learned (RL)

**Pipeline overview:**

```
Reference Path → Find Lookahead Point → Compute Steering (δ) → Bicycle Model → Update State → Repeat
     (from A*)              ↑                                         │
                            └─────────── feedback ────────────────────┘
```

### Control Methods on a Spectrum

| Method | Type | Model? | Key Property |
|--------|------|--------|--------------|
| **PID** | Classical | No | Simple, no model |
| **Pure Pursuit** | Classical geometric | Kinematic | One tunable parameter (lookahead) |
| **Stanley** | Classical geometric | Kinematic | Handles cross-track error directly |
| **LQR** | Optimal | Linear | Provably optimal, constant gain |
| **MPC** | Optimal | Any | Handles constraints |
| **RL** | Learned | None | Flexible, data-hungry |

## From N3’s tracks to action

**N3** left you with a world that is not just *detected* but **tracked and predicted**: every confirmed object carried a state $[x, y, z, v_x, v_y, v_z]$ in a fixed world frame, plus a **prediction arrow** showing where it will be ~0.5 s (5 steps) ahead. N3 ended on the question *“what do we do with all this?”* — this notebook is the first half of the answer: a tracked, predicted world only earns its keep once it becomes a **steering angle and a throttle command**.

Projected onto the BEV ground plane, each track is just `{id, pos [x, y], vel [vx, vy]}`, and N3’s prediction arrow is the constant-velocity forecast $p + v\,t$. That forecast — *where will this object be?* — is what the path-tracker steers around here, and what the safety layer in **N5** brakes for.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A few tracks in the shape N3 hands off (synthetic, so this cell runs standalone).
tracks = [
    {"id": 1, "pos": np.array([12.0,  1.5]), "vel": np.array([ 8.0,  0.0])},
    {"id": 2, "pos": np.array([20.0, -3.5]), "vel": np.array([ 5.0,  1.2])},
    {"id": 3, "pos": np.array([ 6.0,  4.0]), "vel": np.array([10.0, -0.5])},
]

def forecast(track, horizon=2.0, dt=0.2):
    """Constant-velocity rollout: positions p + v*t over [0, horizon]."""
    ts = np.arange(0.0, horizon + 1e-9, dt)
    return track["pos"][None, :] + np.outer(ts, track["vel"])

fig, ax = plt.subplots(figsize=(7, 5))
for tr in tracks:
    fcast = forecast(tr)
    ax.plot(fcast[:, 0], fcast[:, 1], "--", alpha=0.7)
    ax.scatter(*tr["pos"], s=60, label=f"track {tr['id']}")
    ax.annotate(f"  v={np.linalg.norm(tr['vel']):.1f} m/s", tr["pos"])
ax.set_title("Tracked objects and their 2 s constant-velocity forecasts")
ax.set_xlabel("x (m, forward)"); ax.set_ylabel("y (m, left)")
ax.legend(); ax.grid(alpha=0.3); ax.set_aspect("equal")
plt.show()

print("Each track gives us a forecast we can plan against; now we learn to steer.")

## 1. Install and Import Dependencies

In [ ]:
%pip install numpy matplotlib --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.random.seed(42)

print("All libraries loaded successfully!")

## The Kinematic Bicycle Model

A real vehicle has four wheels, suspension, tire slip, and aerodynamic forces — too complex for a first controller. The **kinematic bicycle model** collapses this into two wheels (front and rear axle midpoints) connected by a rigid wheelbase $L$. This is a good approximation at low-to-moderate speeds where tire slip angles are small.

### State and Controls

**State:** $[x, y, \psi, v]$ — rear-axle position, heading (yaw angle), speed

**Controls:** $[\delta, a]$ — front-wheel steering angle, acceleration

### Equations of Motion

$$\dot{x} = v \cos(\psi)$$
$$\dot{y} = v \sin(\psi)$$
$$\dot{\psi} = \frac{v}{L} \tan(\delta)$$
$$\dot{v} = a$$

where $L$ is the wheelbase (distance from rear to front axle).

### Bicycle Geometry

```
          Front Axle
             / δ  (steering angle)
            /
    ───────●─────────
           │         
           │  L (wheelbase)
           │         
    ───────●─────────→ heading ψ
        Rear Axle
       (x, y)
```

The rear axle is the reference point because it moves in the direction the vehicle is facing — no sideslip. The front axle traces a wider arc when turning.

| Parameter | Symbol | Value | Notes |
|-----------|--------|-------|-------|
| Wheelbase | $L$ | 2.5 m | Typical passenger car |
| Max steering | $\delta_{\max}$ | ±35° (0.61 rad) | Mechanical limit |
| Time step | $\Delta t$ | 0.05 s | 20 Hz control loop |
| Cruise speed | $v$ | 5 m/s | ~18 km/h |

In [ ]:
# ---------- Vehicle parameters ----------
L = 2.5             # wheelbase (m)
MAX_STEER = np.radians(35)  # ±35° max steering angle
DT = 0.05           # time step (s) — 20 Hz control loop
V_CRUISE = 5.0      # cruise speed (m/s)


def normalize_angle(angle):
    """Wrap angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi


def bicycle_step(state, delta, a, L, dt):
    """
    Advance kinematic bicycle model by one time step (Euler integration).
    
    Parameters:
        state : [x, y, psi, v]  — rear-axle position, heading, speed
        delta : front-wheel steering angle (rad)
        a     : acceleration (m/s^2)
        L     : wheelbase (m)
        dt    : time step (s)
    
    Returns:
        new state [x, y, psi, v]
    """
    x, y, psi, v = state
    x_new   = x   + v * np.cos(psi) * dt
    y_new   = y   + v * np.sin(psi) * dt
    psi_new = psi  + (v / L) * np.tan(delta) * dt
    v_new   = v   + a * dt
    psi_new = normalize_angle(psi_new)
    return np.array([x_new, y_new, psi_new, v_new])


# ---------- Sanity checks ----------
# Test 1: straight-line driving (delta = 0)
s0 = np.array([0.0, 0.0, 0.0, 5.0])  # heading east at 5 m/s
s1 = bicycle_step(s0, delta=0.0, a=0.0, L=L, dt=1.0)
print("Test 1 — Straight-line (δ=0, heading=0°, v=5 m/s, dt=1s):")
print(f"  Start: x={s0[0]:.1f}, y={s0[1]:.1f}, ψ={np.degrees(s0[2]):.1f}°, v={s0[3]:.1f}")
print(f"  After: x={s1[0]:.1f}, y={s1[1]:.1f}, ψ={np.degrees(s1[2]):.1f}°, v={s1[3]:.1f}")

# Test 2: turning (delta = 20°)
s2 = bicycle_step(s0, delta=np.radians(20), a=0.0, L=L, dt=1.0)
print(f"\nTest 2 — Turning (δ=20°, heading=0°, v=5 m/s, dt=1s):")
print(f"  Start: x={s0[0]:.1f}, y={s0[1]:.1f}, ψ={np.degrees(s0[2]):.1f}°, v={s0[3]:.1f}")
print(f"  After: x={s2[0]:.1f}, y={s2[1]:.1f}, ψ={np.degrees(s2[2]):.1f}°, v={s2[3]:.1f}")

# Test 3: acceleration from rest
s_rest = np.array([0.0, 0.0, np.pi/4, 0.0])  # heading NE, stopped
s3 = bicycle_step(s_rest, delta=0.0, a=2.0, L=L, dt=1.0)
print(f"\nTest 3 — Acceleration from rest (a=2 m/s², heading=45°, dt=1s):")
print(f"  Start: x={s_rest[0]:.1f}, y={s_rest[1]:.1f}, ψ={np.degrees(s_rest[2]):.1f}°, v={s_rest[3]:.1f}")
print(f"  After: x={s3[0]:.1f}, y={s3[1]:.1f}, ψ={np.degrees(s3[2]):.1f}°, v={s3[3]:.1f}")

## 2. Generate a Reference Path

We build a synthetic reference path that exercises the controller through a variety of curvatures:

1. **Straight east** (~50 m) — the baseline; any controller can handle this
2. **Sweeping right turn** (arc, ~80 m) — sustained curvature tests steady-state tracking
3. **Short straight** (~30 m) — transition from curve to straight
4. **Left-right chicane** (S-curve) — rapid curvature reversal, the hardest test
5. **Straight exit** (~40 m) — recovery after the chicane

The path is stored as a dense array of $(x, y)$ waypoints (~600 points) so that the controller can find smooth lookahead targets. In a real pipeline, this path would come from the A* planner in Workshop 4.2 — you could swap the synthetic path for a planned one with no changes to the controller code.

In [ ]:
def generate_reference_path():
    """
    Build a synthetic reference path with varied curvature.
    Returns: path (N, 2) array of [x, y] waypoints.
    """
    ds = 0.5  # spacing between waypoints (m)
    segments = []
    
    # Segment 1: straight east for ~50 m
    n1 = int(50 / ds)
    seg1_x = np.linspace(0, 50, n1)
    seg1_y = np.zeros(n1)
    segments.append(np.column_stack([seg1_x, seg1_y]))
    
    # Segment 2: sweeping right turn (arc, radius ~50 m, ~80 m arc length)
    R_turn = 50.0
    arc_length = 80.0
    theta_span = arc_length / R_turn  # ~1.6 rad (~91°)
    n2 = int(arc_length / ds)
    # Center of the arc is below the endpoint of segment 1
    cx, cy = 50.0, -R_turn
    thetas = np.linspace(np.pi / 2, np.pi / 2 - theta_span, n2)
    seg2_x = cx + R_turn * np.cos(thetas)
    seg2_y = cy + R_turn * np.sin(thetas)
    segments.append(np.column_stack([seg2_x, seg2_y]))
    
    # Get the heading at end of arc for connecting the next segment
    end_heading = -theta_span  # heading after the right turn
    end_x, end_y = seg2_x[-1], seg2_y[-1]
    
    # Segment 3: short straight (~30 m) continuing in the arc-exit direction
    n3 = int(30 / ds)
    seg3_x = end_x + np.linspace(0, 30 * np.cos(end_heading), n3)
    seg3_y = end_y + np.linspace(0, 30 * np.sin(end_heading), n3)
    segments.append(np.column_stack([seg3_x, seg3_y]))
    
    end_x2, end_y2 = seg3_x[-1], seg3_y[-1]
    
    # Segment 4: chicane (S-curve) — left then right
    R_chic = 30.0
    chic_arc = 25.0
    theta_chic = chic_arc / R_chic
    n4 = int(chic_arc / ds)
    
    # Left turn arc (center is to the left of current heading)
    perp_left = end_heading + np.pi / 2
    cx_l = end_x2 + R_chic * np.cos(perp_left)
    cy_l = end_y2 + R_chic * np.sin(perp_left)
    start_angle_l = perp_left + np.pi  # angle from center to current point
    thetas_l = np.linspace(start_angle_l, start_angle_l + theta_chic, n4)
    seg4a_x = cx_l + R_chic * np.cos(thetas_l)
    seg4a_y = cy_l + R_chic * np.sin(thetas_l)
    segments.append(np.column_stack([seg4a_x, seg4a_y]))
    
    # Right turn arc (center is to the right)
    mid_heading = end_heading + theta_chic
    end_x3, end_y3 = seg4a_x[-1], seg4a_y[-1]
    perp_right = mid_heading - np.pi / 2
    cx_r = end_x3 + R_chic * np.cos(perp_right)
    cy_r = end_y3 + R_chic * np.sin(perp_right)
    start_angle_r = perp_right + np.pi
    thetas_r = np.linspace(start_angle_r, start_angle_r - theta_chic, n4)
    seg4b_x = cx_r + R_chic * np.cos(thetas_r)
    seg4b_y = cy_r + R_chic * np.sin(thetas_r)
    segments.append(np.column_stack([seg4b_x, seg4b_y]))
    
    # Segment 5: straight exit (~40 m)
    exit_heading = end_heading  # chicane returns to original heading
    end_x4, end_y4 = seg4b_x[-1], seg4b_y[-1]
    n5 = int(40 / ds)
    seg5_x = end_x4 + np.linspace(0, 40 * np.cos(exit_heading), n5)
    seg5_y = end_y4 + np.linspace(0, 40 * np.sin(exit_heading), n5)
    segments.append(np.column_stack([seg5_x, seg5_y]))
    
    # Concatenate and remove duplicate junction points
    path = np.vstack(segments)
    
    # Remove near-duplicate consecutive points
    diffs = np.sqrt(np.sum(np.diff(path, axis=0)**2, axis=1))
    keep = np.concatenate([[True], diffs > 0.01])
    path = path[keep]
    
    return path


ref_path = generate_reference_path()

# Compute path length
path_diffs = np.diff(ref_path, axis=0)
path_seg_lengths = np.sqrt(path_diffs[:, 0]**2 + path_diffs[:, 1]**2)
path_total_length = np.sum(path_seg_lengths)

print(f"Reference path: {len(ref_path)} waypoints, {path_total_length:.1f} m total length")
print(f"X range: [{ref_path[:, 0].min():.1f}, {ref_path[:, 0].max():.1f}] m")
print(f"Y range: [{ref_path[:, 1].min():.1f}, {ref_path[:, 1].max():.1f}] m")

# Visualize the reference path
fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(ref_path[:, 0], ref_path[:, 1], "k-", linewidth=2, alpha=0.6, label="Reference path")
ax.plot(ref_path[0, 0], ref_path[0, 1], "go", markersize=12, zorder=5, label="Start")
ax.plot(ref_path[-1, 0], ref_path[-1, 1], "rs", markersize=12, zorder=5, label="End")

# Mark segment transitions with evenly-spaced arrows
arrow_spacing = max(1, len(ref_path) // 20)
for i in range(0, len(ref_path) - 1, arrow_spacing):
    dx = ref_path[i + 1, 0] - ref_path[i, 0]
    dy = ref_path[i + 1, 1] - ref_path[i, 1]
    ax.annotate("", xy=(ref_path[i, 0] + dx, ref_path[i, 1] + dy),
                xytext=(ref_path[i, 0], ref_path[i, 1]),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))

ax.set_xlabel("X (m)", fontsize=12)
ax.set_ylabel("Y (m)", fontsize=12)
ax.set_title("Reference Path: Straight → Right Turn → Chicane → Straight Exit", fontsize=14)
ax.set_aspect("equal")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Pure Pursuit Controller

Pure pursuit is the simplest geometric path-tracking controller. The idea:

1. **Pick a lookahead point** on the reference path at distance $L_d$ ahead of the vehicle
2. **Steer toward it** along a circular arc that passes through both the rear axle and the lookahead point

### The Geometry

Given the vehicle at $(x, y)$ with heading $\psi$ and a lookahead point at $(x_L, y_L)$:

1. Compute $\alpha$ — the angle from the vehicle's heading to the lookahead point:
$$\alpha = \text{atan2}(y_L - y,\; x_L - x) - \psi$$

2. Compute the actual lookahead distance $L_d = \sqrt{(x_L - x)^2 + (y_L - y)^2}$

3. Compute the steering angle:
$$\delta = \arctan\left(\frac{2 \, L \, \sin(\alpha)}{L_d}\right)$$

where $L$ is the wheelbase.

### Why This Works

The formula comes from fitting a circular arc through the rear axle and the lookahead point. The arc's curvature $\kappa = \frac{2 \sin(\alpha)}{L_d}$, and the bicycle model relates curvature to steering: $\kappa = \frac{\tan(\delta)}{L}$.

### The Lookahead Distance $L_d$

This is the **only tunable parameter** in pure pursuit:

| $L_d$ | Behavior | Problem |
|-------|----------|---------|
| **Short** (e.g. 3 m) | Aggressive, responsive | Oscillates, overshoots corners |
| **Balanced** (e.g. 8 m) | Smooth, accurate | Sweet spot |
| **Long** (e.g. 20 m) | Very smooth | Cuts corners, high cross-track error in curves |

In [ ]:
def find_lookahead_point(state, path, lookahead_dist):
    """
    Find the point on the path at approximately lookahead_dist ahead of the vehicle.
    
    Strategy: find the nearest path point, then walk forward along the path until
    the Euclidean distance from the vehicle exceeds lookahead_dist.
    
    Parameters:
        state          : [x, y, psi, v]
        path           : (N, 2) array of [x, y] waypoints
        lookahead_dist : target lookahead distance (m)
    
    Returns:
        lookahead_point : [x, y] on the path
        nearest_idx     : index of the nearest path point
    """
    x, y = state[0], state[1]
    
    # Find nearest path point
    dists = np.sqrt((path[:, 0] - x)**2 + (path[:, 1] - y)**2)
    nearest_idx = np.argmin(dists)
    
    # Walk forward from nearest point until distance >= lookahead_dist
    lookahead_point = path[-1]  # default to end of path
    for i in range(nearest_idx, len(path)):
        dist = np.sqrt((path[i, 0] - x)**2 + (path[i, 1] - y)**2)
        if dist >= lookahead_dist:
            lookahead_point = path[i]
            break
    
    return lookahead_point, nearest_idx


def pure_pursuit_steering(state, lookahead_point, L):
    """
    Compute the steering angle to reach the lookahead point using pure pursuit.
    
    Parameters:
        state           : [x, y, psi, v]
        lookahead_point : [x, y] target on the path
        L               : wheelbase (m)
    
    Returns:
        delta : steering angle (rad), clipped to [-MAX_STEER, MAX_STEER]
    """
    x, y, psi, v = state
    lx, ly = lookahead_point
    
    # Angle from vehicle to lookahead point (world frame)
    angle_to_target = np.arctan2(ly - y, lx - x)
    
    # Alpha: angle between vehicle heading and direction to lookahead point
    alpha = normalize_angle(angle_to_target - psi)
    
    # Lookahead distance (actual)
    Ld = np.sqrt((lx - x)**2 + (ly - y)**2)
    Ld = max(Ld, 0.1)  # avoid division by zero
    
    # Pure pursuit steering law
    delta = np.arctan2(2.0 * L * np.sin(alpha), Ld)
    
    # Clip to mechanical limits
    delta = np.clip(delta, -MAX_STEER, MAX_STEER)
    
    return delta


# ---------- Quick test ----------
test_state = np.array([0.0, 0.0, 0.0, 5.0])  # heading east
test_la = np.array([10.0, 2.0])  # lookahead slightly to the left
test_delta = pure_pursuit_steering(test_state, test_la, L)
print(f"Test: vehicle at origin heading east, lookahead at (10, 2)")
print(f"  Steering angle: {np.degrees(test_delta):.2f}° (should steer left/positive)")

test_la2 = np.array([10.0, -2.0])  # lookahead slightly to the right
test_delta2 = pure_pursuit_steering(test_state, test_la2, L)
print(f"  Lookahead at (10, -2): δ = {np.degrees(test_delta2):.2f}° (should steer right/negative)")

test_la3 = np.array([10.0, 0.0])  # lookahead straight ahead
test_delta3 = pure_pursuit_steering(test_state, test_la3, L)
print(f"  Lookahead at (10,  0): δ = {np.degrees(test_delta3):.2f}° (should be ~0)")

## 4. Run Pure Pursuit on the Reference Path

We run the full simulation loop with a balanced lookahead distance ($L_d = 8$ m):

1. At each time step, find the lookahead point on the reference path
2. Compute the steering angle via pure pursuit
3. Step the bicycle model forward
4. Record the trajectory, steering history, and cross-track error

We use constant speed ($v = 5$ m/s) — no acceleration controller, just holding speed.

In [ ]:
def run_pure_pursuit(path, lookahead_dist, v_target=V_CRUISE, L=L, dt=DT,
                     max_steps=20000, start_state=None):
    """
    Run pure pursuit simulation on the given path.
    
    Returns:
        trajectory  : (N, 4) array of states [x, y, psi, v]
        steerings   : (N,) array of steering angles
        cte_history : (N,) cross-track error at each step
        la_points   : (N, 2) lookahead points at each step
    """
    if start_state is None:
        # Start at the beginning of the path, heading toward the second point
        dx = path[1, 0] - path[0, 0]
        dy = path[1, 1] - path[0, 1]
        heading0 = np.arctan2(dy, dx)
        start_state = np.array([path[0, 0], path[0, 1], heading0, v_target])
    
    state = start_state.copy()
    trajectory = [state.copy()]
    steerings = []
    cte_history = []
    la_points = []
    
    for step in range(max_steps):
        # Find lookahead point
        la_pt, nearest_idx = find_lookahead_point(state, path, lookahead_dist)
        la_points.append(la_pt.copy())
        
        # Compute cross-track error (signed distance to nearest path point)
        nearest_pt = path[nearest_idx]
        # Signed CTE: positive if vehicle is to the left of the path
        if nearest_idx < len(path) - 1:
            path_dx = path[nearest_idx + 1, 0] - path[nearest_idx, 0]
            path_dy = path[nearest_idx + 1, 1] - path[nearest_idx, 1]
        else:
            path_dx = path[nearest_idx, 0] - path[nearest_idx - 1, 0]
            path_dy = path[nearest_idx, 1] - path[nearest_idx - 1, 1]
        # Cross product gives signed perpendicular distance
        ex = state[0] - nearest_pt[0]
        ey = state[1] - nearest_pt[1]
        cte = (path_dx * ey - path_dy * ex) / (np.sqrt(path_dx**2 + path_dy**2) + 1e-9)
        cte_history.append(cte)
        
        # Compute steering
        delta = pure_pursuit_steering(state, la_pt, L)
        steerings.append(delta)
        
        # Simple speed controller: hold constant speed
        a = 2.0 * (v_target - state[3])  # proportional speed control
        
        # Step the bicycle model
        state = bicycle_step(state, delta, a, L, dt)
        trajectory.append(state.copy())
        
        # Stop when we're near the end of the path
        dist_to_end = np.sqrt((state[0] - path[-1, 0])**2 + (state[1] - path[-1, 1])**2)
        if dist_to_end < 2.0:
            break
    
    trajectory = np.array(trajectory)
    steerings = np.array(steerings)
    cte_history = np.array(cte_history)
    la_points = np.array(la_points)
    
    return trajectory, steerings, cte_history, la_points


# ---------- Run with balanced lookahead ----------
Ld_balanced = 8.0
traj_b, steer_b, cte_b, la_b = run_pure_pursuit(ref_path, Ld_balanced)

# Compute stats
traj_length = np.sum(np.sqrt(np.diff(traj_b[:, 0])**2 + np.diff(traj_b[:, 1])**2))
rmse_cte = np.sqrt(np.mean(cte_b**2))
max_cte = np.max(np.abs(cte_b))
mean_abs_steer = np.mean(np.abs(steer_b))

print(f"Pure Pursuit Simulation (Ld = {Ld_balanced} m)")
print(f"{'='*50}")
print(f"  Simulation steps:     {len(steer_b)}")
print(f"  Simulation time:      {len(steer_b) * DT:.1f} s")
print(f"  Path length (ref):    {path_total_length:.1f} m")
print(f"  Trajectory length:    {traj_length:.1f} m")
print(f"  Cross-track RMSE:     {rmse_cte:.3f} m")
print(f"  Max |CTE|:            {max_cte:.3f} m")
print(f"  Mean |steering|:      {np.degrees(mean_abs_steer):.2f}°")
print(f"  Max |steering|:       {np.degrees(np.max(np.abs(steer_b))):.2f}°")

## 5. Animated Pure Pursuit

We animate the simulation showing:
- **Left panel**: the vehicle moving along the path with the lookahead point and traced trajectory
- **Right panel**: cross-track error and steering angle over time

The vehicle is drawn as a triangle oriented by heading. The red dot is the lookahead point, with a line connecting it to the vehicle.

In [ ]:
def make_vehicle_triangle(x, y, psi, size=2.0):
    """Create a triangle polygon centered at (x, y) with heading psi."""
    # Triangle points (front, back-left, back-right) in vehicle frame
    pts = np.array([[size, 0],
                    [-size * 0.5, size * 0.5],
                    [-size * 0.5, -size * 0.5]])
    # Rotate by heading
    R = np.array([[np.cos(psi), -np.sin(psi)],
                  [np.sin(psi),  np.cos(psi)]])
    pts_rot = (R @ pts.T).T
    pts_rot[:, 0] += x
    pts_rot[:, 1] += y
    return pts_rot


# Subsample frames for animation (aim for ~150 frames)
n_sim_steps = len(steer_b)
anim_step = max(1, n_sim_steps // 150)
anim_indices = list(range(0, n_sim_steps, anim_step))

fig, (ax_map, ax_diag) = plt.subplots(1, 2, figsize=(18, 7), dpi=80,
                                       gridspec_kw={"width_ratios": [3, 2]})

# --- Left: map view ---
pad = 10
ax_map.set_xlim(ref_path[:, 0].min() - pad, ref_path[:, 0].max() + pad)
ax_map.set_ylim(ref_path[:, 1].min() - pad, ref_path[:, 1].max() + pad)
ax_map.set_aspect("equal")
ax_map.set_xlabel("X (m)", fontsize=12)
ax_map.set_ylabel("Y (m)", fontsize=12)
ax_map.set_title(f"Pure Pursuit (Ld = {Ld_balanced} m)", fontsize=14)
ax_map.grid(True, alpha=0.3)

ax_map.plot(ref_path[:, 0], ref_path[:, 1], color="gray", linewidth=2,
            alpha=0.5, label="Reference path")

traj_line, = ax_map.plot([], [], "b-", linewidth=1.5, label="Vehicle trajectory")
la_dot, = ax_map.plot([], [], "ro", markersize=8, zorder=6, label="Lookahead point")
la_line, = ax_map.plot([], [], "r-", linewidth=1, alpha=0.5)
vehicle_poly = plt.Polygon(make_vehicle_triangle(0, 0, 0), closed=True,
                           facecolor="blue", edgecolor="darkblue",
                           linewidth=1.5, zorder=7, alpha=0.8)
ax_map.add_patch(vehicle_poly)
ax_map.legend(fontsize=9, loc="upper left")

# --- Right: diagnostics ---
time_arr = np.arange(n_sim_steps) * DT

ax_cte = ax_diag
ax_cte.set_xlabel("Time (s)", fontsize=12)
ax_cte.set_ylabel("Cross-Track Error (m)", fontsize=12, color="tab:blue")
ax_cte.set_title("CTE and Steering Over Time", fontsize=14)
ax_cte.set_xlim(0, time_arr[-1])
max_cte_plot = max(np.abs(cte_b).max() * 1.2, 1.0)
ax_cte.set_ylim(-max_cte_plot, max_cte_plot)
ax_cte.axhline(0, color="gray", linewidth=0.5)
ax_cte.grid(True, alpha=0.3)

cte_line, = ax_cte.plot([], [], "tab:blue", linewidth=1.5, label="CTE")
ax_cte.tick_params(axis="y", labelcolor="tab:blue")

ax_steer = ax_cte.twinx()
ax_steer.set_ylabel("Steering Angle (deg)", fontsize=12, color="tab:orange")
max_steer_plot = np.degrees(np.abs(steer_b).max()) * 1.2
ax_steer.set_ylim(-max_steer_plot, max_steer_plot)
steer_line, = ax_steer.plot([], [], "tab:orange", linewidth=1.5, alpha=0.7, label="Steering")
ax_steer.tick_params(axis="y", labelcolor="tab:orange")

time_vline = ax_cte.axvline(0, color="gray", linewidth=1, alpha=0.4, linestyle="--")

# Combine legends
lines = [cte_line, steer_line]
labels = [l.get_label() for l in lines]
ax_cte.legend(lines, labels, fontsize=10, loc="upper right")

plt.tight_layout()


def update_anim(frame):
    k = anim_indices[frame]
    s = slice(0, k + 1)
    
    # Update trajectory trace
    traj_line.set_data(traj_b[s, 0], traj_b[s, 1])
    
    # Update vehicle triangle
    tri = make_vehicle_triangle(traj_b[k, 0], traj_b[k, 1], traj_b[k, 2])
    vehicle_poly.set_xy(tri)
    
    # Update lookahead point and line
    la_dot.set_data([la_b[k, 0]], [la_b[k, 1]])
    la_line.set_data([traj_b[k, 0], la_b[k, 0]], [traj_b[k, 1], la_b[k, 1]])
    
    # Update diagnostics
    t = time_arr[s]
    cte_line.set_data(t, cte_b[s])
    steer_line.set_data(t, np.degrees(steer_b[s]))
    time_vline.set_xdata([time_arr[k], time_arr[k]])
    
    return (traj_line, vehicle_poly, la_dot, la_line, cte_line, steer_line, time_vline)


anim_pp = FuncAnimation(fig, update_anim, frames=len(anim_indices),
                        interval=40, blit=False)
plt.close(fig)

print(f"Animation: {len(anim_indices)} frames  |  Press the play button to start")
display(HTML(anim_pp.to_jshtml()))

## 6. Lookahead Distance Sensitivity Sweep

The lookahead distance $L_d$ is pure pursuit's only tunable parameter. Its effect:

- **Short $L_d$ (3 m)**: the controller reacts aggressively to nearby path curvature. Good responsiveness, but oscillates around the path — especially on straights or exiting curves.
- **Balanced $L_d$ (8 m)**: smooth tracking with low cross-track error. The sweet spot for moderate speeds.
- **Long $L_d$ (20 m)**: very smooth steering, but the controller "looks too far ahead" and cuts corners on tight curves.

We run three simulations and compare them side by side.

In [ ]:
# ---------- Run 3 simulations with different lookahead distances ----------
Ld_configs = [
    (3.0,  "Short (3 m)",    "tab:red"),
    (8.0,  "Balanced (8 m)", "tab:blue"),
    (20.0, "Long (20 m)",    "tab:green"),
]

results = []
for Ld_val, label, color in Ld_configs:
    traj, steer, cte, la = run_pure_pursuit(ref_path, Ld_val)
    rmse = np.sqrt(np.mean(cte**2))
    max_abs_cte = np.max(np.abs(cte))
    results.append({
        "Ld": Ld_val,
        "label": label,
        "color": color,
        "traj": traj,
        "steer": steer,
        "cte": cte,
        "la": la,
        "rmse": rmse,
        "max_cte": max_abs_cte,
    })
    print(f"  Ld = {Ld_val:5.1f} m  |  RMSE CTE = {rmse:.3f} m  |  Max |CTE| = {max_abs_cte:.3f} m  |  Steps = {len(steer)}")

# ---------- Static comparison plot ----------
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: trajectories overlaid on reference path
ax = axes[0]
ax.plot(ref_path[:, 0], ref_path[:, 1], "k-", linewidth=3, alpha=0.3, label="Reference")
for r in results:
    ax.plot(r["traj"][:, 0], r["traj"][:, 1], color=r["color"], linewidth=2,
            label=f"{r['label']} (RMSE={r['rmse']:.3f} m)")
ax.set_xlabel("X (m)", fontsize=12)
ax.set_ylabel("Y (m)", fontsize=12)
ax.set_title("Trajectory Comparison: Lookahead Distance Sweep", fontsize=14)
ax.set_aspect("equal")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: cross-track error over time
ax = axes[1]
for r in results:
    t = np.arange(len(r["cte"])) * DT
    ax.plot(t, r["cte"], color=r["color"], linewidth=1.5, alpha=0.8,
            label=r["label"])
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Cross-Track Error (m)", fontsize=12)
ax.set_title("Cross-Track Error Over Time", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---------- Animated side-by-side comparison ----------

# Find the minimum number of steps across all runs
min_steps = min(len(r["steer"]) for r in results)
anim_step_cmp = max(1, min_steps // 150)
anim_idx_cmp = list(range(0, min_steps, anim_step_cmp))

fig_cmp, axes_cmp = plt.subplots(1, 3, figsize=(20, 7), dpi=80)

artists_cmp = []
for i, (ax, r) in enumerate(zip(axes_cmp, results)):
    ax.plot(ref_path[:, 0], ref_path[:, 1], color="gray", linewidth=2, alpha=0.4)
    pad = 10
    ax.set_xlim(ref_path[:, 0].min() - pad, ref_path[:, 0].max() + pad)
    ax.set_ylim(ref_path[:, 1].min() - pad, ref_path[:, 1].max() + pad)
    ax.set_aspect("equal")
    ax.set_title(f"{r['label']}\nRMSE = {r['rmse']:.3f} m", fontsize=13)
    ax.set_xlabel("X (m)", fontsize=11)
    if i == 0:
        ax.set_ylabel("Y (m)", fontsize=11)
    ax.grid(True, alpha=0.3)
    
    trail, = ax.plot([], [], color=r["color"], linewidth=1.5)
    la_d, = ax.plot([], [], "ro", markersize=6, zorder=6)
    la_l, = ax.plot([], [], "r-", linewidth=1, alpha=0.4)
    veh = plt.Polygon(make_vehicle_triangle(0, 0, 0), closed=True,
                      facecolor=r["color"], edgecolor="black",
                      linewidth=1, zorder=7, alpha=0.8)
    ax.add_patch(veh)
    
    # CTE text
    cte_txt = ax.text(0.02, 0.02, "", transform=ax.transAxes, fontsize=10,
                      fontweight="bold", va="bottom",
                      bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    artists_cmp.append((trail, la_d, la_l, veh, cte_txt, r))

plt.tight_layout()


def update_cmp(frame):
    k = anim_idx_cmp[frame]
    s = slice(0, k + 1)
    
    for trail, la_d, la_l, veh, cte_txt, r in artists_cmp:
        if k < len(r["steer"]):
            trail.set_data(r["traj"][s, 0], r["traj"][s, 1])
            tri = make_vehicle_triangle(r["traj"][k, 0], r["traj"][k, 1], r["traj"][k, 2])
            veh.set_xy(tri)
            la_d.set_data([r["la"][k, 0]], [r["la"][k, 1]])
            la_l.set_data([r["traj"][k, 0], r["la"][k, 0]],
                         [r["traj"][k, 1], r["la"][k, 1]])
            cte_txt.set_text(f"CTE: {r['cte'][k]:.3f} m")
    
    return tuple(item for group in artists_cmp for item in group[:5])


anim_cmp = FuncAnimation(fig_cmp, update_cmp, frames=len(anim_idx_cmp),
                         interval=40, blit=False)
plt.close(fig_cmp)

print(f"Animation: {len(anim_idx_cmp)} frames — three lookahead distances side by side")
print("Watch: short Ld oscillates, balanced tracks well, long Ld cuts corners")
display(HTML(anim_cmp.to_jshtml()))

## 7. Cross-Track Error Analysis

We summarize the cross-track error behavior for all three lookahead settings with:
1. CTE over path progress for each setting
2. RMSE bar chart comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: CTE over distance along path
ax = axes[0]
for r in results:
    # Compute cumulative distance along the vehicle trajectory
    traj_dists = np.cumsum(np.sqrt(np.diff(r["traj"][:len(r["cte"]), 0])**2 +
                                   np.diff(r["traj"][:len(r["cte"]), 1])**2))
    traj_dists = np.concatenate([[0], traj_dists])
    ax.plot(traj_dists, r["cte"], color=r["color"], linewidth=1.5, alpha=0.8,
            label=r["label"])
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Distance Along Path (m)", fontsize=12)
ax.set_ylabel("Cross-Track Error (m)", fontsize=12)
ax.set_title("Cross-Track Error vs Path Progress", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: RMSE bar chart
ax = axes[1]
labels_bar = [r["label"] for r in results]
rmse_vals = [r["rmse"] for r in results]
max_cte_vals = [r["max_cte"] for r in results]
colors_bar = [r["color"] for r in results]

x_pos = np.arange(len(results))
width = 0.35

bars1 = ax.bar(x_pos - width/2, rmse_vals, width, color=colors_bar, alpha=0.8, label="RMSE CTE")
bars2 = ax.bar(x_pos + width/2, max_cte_vals, width, color=colors_bar, alpha=0.4,
               edgecolor=colors_bar, linewidth=2, label="Max |CTE|")

# Add value labels on bars
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
            f"{h:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
            f"{h:.3f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x_pos)
ax.set_xticklabels(labels_bar, fontsize=11)
ax.set_ylabel("Error (m)", fontsize=12)
ax.set_title("Cross-Track Error Summary", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nCross-Track Error Summary:")
print(f"{'Setting':<20} {'RMSE (m)':>10} {'Max |CTE| (m)':>15}")
print("-" * 48)
for r in results:
    print(f"{r['label']:<20} {r['rmse']:>10.4f} {r['max_cte']:>15.4f}")

## 8. Exercise: Stanley Controller (Optional)

The **Stanley controller** was developed at Stanford for their DARPA Grand Challenge entry ("Stanley", 2005 — the first autonomous vehicle to finish the race). It uses a different geometry than pure pursuit:

### Key Differences from Pure Pursuit

| Feature | Pure Pursuit | Stanley |
|---------|-------------|--------|
| Reference point | Rear axle | **Front axle** |
| Error signal | Angle to lookahead point | **Heading error + cross-track error** |
| Tunable parameter | Lookahead distance $L_d$ | Cross-track gain $k$ |
| Corner behavior | Cuts corners (with large $L_d$) | Handles cross-track error directly |

### Stanley Steering Law

The steering angle is the sum of two terms:

$$\delta = (\psi_{\text{path}} - \psi) + \arctan\left(\frac{k \, e}{v}\right)$$

where:
- $\psi_{\text{path}} - \psi$ corrects the **heading error** (align with the path tangent)
- $\arctan(k \, e / v)$ corrects the **cross-track error** $e$ proportionally, with gain $k$
- Dividing by $v$ makes the correction less aggressive at higher speeds (stability)

### Why Stanley Can Outperform Pure Pursuit

Pure pursuit only steers toward a target point — it has no explicit notion of how far off the path the vehicle is. Stanley directly penalizes cross-track error, so it converges to the path more predictably, especially at speed through corners.

In [ ]:
def stanley_steering(state, path, k_gain, L_wheelbase):
    """
    Compute steering angle using the Stanley controller.
    
    Uses the front-axle position as the reference point.
    
    Parameters:
        state        : [x, y, psi, v] (rear-axle position)
        path         : (N, 2) waypoints
        k_gain       : cross-track error gain
        L_wheelbase  : wheelbase (m)
    
    Returns:
        delta : steering angle (rad)
    """
    x, y, psi, v = state
    
    # Compute front-axle position
    fx = x + L_wheelbase * np.cos(psi)
    fy = y + L_wheelbase * np.sin(psi)
    
    # Find nearest path point to the front axle
    dists = np.sqrt((path[:, 0] - fx)**2 + (path[:, 1] - fy)**2)
    nearest_idx = np.argmin(dists)
    
    # Path heading at the nearest point
    if nearest_idx < len(path) - 1:
        path_dx = path[nearest_idx + 1, 0] - path[nearest_idx, 0]
        path_dy = path[nearest_idx + 1, 1] - path[nearest_idx, 1]
    else:
        path_dx = path[nearest_idx, 0] - path[nearest_idx - 1, 0]
        path_dy = path[nearest_idx, 1] - path[nearest_idx - 1, 1]
    psi_path = np.arctan2(path_dy, path_dx)
    
    # Heading error
    heading_error = normalize_angle(psi_path - psi)
    
    # Cross-track error (signed): positive if front axle is to the left of the path
    nearest_pt = path[nearest_idx]
    ex = fx - nearest_pt[0]
    ey = fy - nearest_pt[1]
    cte = (path_dx * ey - path_dy * ex) / (np.sqrt(path_dx**2 + path_dy**2) + 1e-9)
    
    # Stanley steering law
    # Note: we use -cte because we want to steer toward the path
    crosstrack_term = np.arctan2(k_gain * (-cte), max(abs(v), 0.5))
    
    delta = heading_error + crosstrack_term
    delta = np.clip(delta, -MAX_STEER, MAX_STEER)
    
    return delta, cte


def run_stanley(path, k_gain, v_target=V_CRUISE, L=L, dt=DT,
                max_steps=20000, start_state=None):
    """
    Run Stanley controller simulation on the given path.
    """
    if start_state is None:
        dx = path[1, 0] - path[0, 0]
        dy = path[1, 1] - path[0, 1]
        heading0 = np.arctan2(dy, dx)
        start_state = np.array([path[0, 0], path[0, 1], heading0, v_target])
    
    state = start_state.copy()
    trajectory = [state.copy()]
    steerings = []
    cte_history = []
    
    for step in range(max_steps):
        delta, cte = stanley_steering(state, path, k_gain, L)
        steerings.append(delta)
        cte_history.append(cte)
        
        # Speed control
        a = 2.0 * (v_target - state[3])
        
        state = bicycle_step(state, delta, a, L, dt)
        trajectory.append(state.copy())
        
        dist_to_end = np.sqrt((state[0] - path[-1, 0])**2 + (state[1] - path[-1, 1])**2)
        if dist_to_end < 2.0:
            break
    
    return (np.array(trajectory), np.array(steerings),
            np.array(cte_history))


# ---------- Run Stanley with a tuned gain ----------
k_stanley = 2.5
traj_s, steer_s, cte_s = run_stanley(ref_path, k_stanley)

rmse_stanley = np.sqrt(np.mean(cte_s**2))
max_cte_stanley = np.max(np.abs(cte_s))

print(f"Stanley Controller (k = {k_stanley})")
print(f"{'='*50}")
print(f"  Simulation steps:     {len(steer_s)}")
print(f"  Simulation time:      {len(steer_s) * DT:.1f} s")
print(f"  Cross-track RMSE:     {rmse_stanley:.4f} m")
print(f"  Max |CTE|:            {max_cte_stanley:.4f} m")
print(f"  Mean |steering|:      {np.degrees(np.mean(np.abs(steer_s))):.2f}°")
print(f"\nComparison with Pure Pursuit (Ld = {Ld_balanced} m):")
print(f"  PP  RMSE = {results[1]['rmse']:.4f} m  |  Stanley RMSE = {rmse_stanley:.4f} m")
if rmse_stanley < results[1]["rmse"]:
    print(f"  Stanley wins by {(1 - rmse_stanley / results[1]['rmse']) * 100:.1f}%")
else:
    print(f"  Pure Pursuit wins by {(1 - results[1]['rmse'] / rmse_stanley) * 100:.1f}%")

In [ ]:
# ---------- Comparison plot: Pure Pursuit vs Stanley ----------

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Top-left: trajectory comparison
ax = axes[0, 0]
ax.plot(ref_path[:, 0], ref_path[:, 1], "k-", linewidth=3, alpha=0.3, label="Reference")
ax.plot(traj_b[:, 0], traj_b[:, 1], "tab:blue", linewidth=2,
        label=f"Pure Pursuit (Ld={Ld_balanced} m)")
ax.plot(traj_s[:, 0], traj_s[:, 1], "tab:purple", linewidth=2, linestyle="--",
        label=f"Stanley (k={k_stanley})")
ax.set_xlabel("X (m)", fontsize=12)
ax.set_ylabel("Y (m)", fontsize=12)
ax.set_title("Trajectory Comparison", fontsize=14)
ax.set_aspect("equal")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Top-right: CTE over time
ax = axes[0, 1]
t_pp = np.arange(len(cte_b)) * DT
t_st = np.arange(len(cte_s)) * DT
ax.plot(t_pp, cte_b, "tab:blue", linewidth=1.5, alpha=0.8, label="Pure Pursuit")
ax.plot(t_st, cte_s, "tab:purple", linewidth=1.5, alpha=0.8, label="Stanley")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Cross-Track Error (m)", fontsize=12)
ax.set_title("Cross-Track Error Over Time", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Bottom-left: steering comparison
ax = axes[1, 0]
ax.plot(t_pp, np.degrees(steer_b), "tab:blue", linewidth=1.5, alpha=0.8, label="Pure Pursuit")
ax.plot(t_st, np.degrees(steer_s), "tab:purple", linewidth=1.5, alpha=0.8, label="Stanley")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Steering Angle (deg)", fontsize=12)
ax.set_title("Steering Angle Over Time", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Bottom-right: RMSE bar chart
ax = axes[1, 1]
controller_labels = [f"Pure Pursuit\n(Ld={Ld_balanced} m)", f"Stanley\n(k={k_stanley})"]
rmse_vals_cmp = [results[1]["rmse"], rmse_stanley]
max_cte_cmp = [results[1]["max_cte"], max_cte_stanley]
bar_colors = ["tab:blue", "tab:purple"]

x_pos = np.arange(2)
width = 0.35
bars1 = ax.bar(x_pos - width/2, rmse_vals_cmp, width, color=bar_colors, alpha=0.8, label="RMSE CTE")
bars2 = ax.bar(x_pos + width/2, max_cte_cmp, width, color=bar_colors, alpha=0.4,
               edgecolor=bar_colors, linewidth=2, label="Max |CTE|")

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.005,
            f"{h:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.005,
            f"{h:.4f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x_pos)
ax.set_xticklabels(controller_labels, fontsize=11)
ax.set_ylabel("Error (m)", fontsize=12)
ax.set_title("Controller Comparison", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nController Comparison Summary:")
print(f"{'Metric':<25} {'Pure Pursuit':>15} {'Stanley':>15}")
print("-" * 58)
print(f"{'RMSE CTE (m)':<25} {results[1]['rmse']:>15.4f} {rmse_stanley:>15.4f}")
print(f"{'Max |CTE| (m)':<25} {results[1]['max_cte']:>15.4f} {max_cte_stanley:>15.4f}")
print(f"{'Mean |steer| (deg)':<25} {np.degrees(np.mean(np.abs(steer_b))):>15.2f} {np.degrees(np.mean(np.abs(steer_s))):>15.2f}")

## Summary

In this notebook we closed the planning-to-action loop by driving a kinematic vehicle along a planned path:

1. **Kinematic bicycle model** — implemented a tractable 4-state vehicle model ($[x, y, \psi, v]$) with front-wheel steering and wheelbase $L = 2.5$ m
2. **Synthetic reference path** — built a varied course (straight, sweeping turn, chicane, straight exit) that can be swapped for A* output from Workshop 4.2
3. **Pure pursuit controller** — implemented geometric path tracking with a single tunable parameter: the lookahead distance $L_d$
4. **Lookahead sensitivity sweep** — demonstrated the tradeoff: short $L_d$ (3 m) oscillates, balanced (8 m) tracks accurately, long (20 m) cuts corners
5. **Stanley controller** — implemented the DARPA Grand Challenge controller that uses heading error + cross-track error feedback for more direct path correction
6. **Head-to-head comparison** — compared pure pursuit and Stanley on the same path with RMSE, max CTE, and steering effort metrics

### Key Takeaways

- The **kinematic bicycle model** is a simple but effective vehicle approximation at moderate speeds — it captures the essential relationship between steering angle and turning radius
- Pure pursuit has **one parameter** ($L_d$) — simple to tune, but the stability/accuracy tradeoff means no single setting is optimal everywhere
- **Short lookahead = oscillation**, long lookahead = corner cutting, balanced = sweet spot — this is the fundamental tradeoff in geometric controllers
- The **Stanley controller** adds explicit cross-track error feedback, which can improve cornering at speed compared to pure pursuit
- **Classical controllers are transparent and analyzable** — you can prove stability bounds, predict failure modes, and explain every steering decision. This matters for safety-critical systems where RL black boxes are hard to certify
- This completes the **sensing → estimation → planning → control** pipeline from Workshops 4.2 and 4.3: camera/LiDAR fusion (N1) → Kalman filtering (N2) → object tracking (N3) → path planning (WS4.2 A*) → path execution (this notebook)

### What's Next

The control spectrum extends well beyond classical geometric controllers:

- **LQR** (Linear Quadratic Regulator) — provably optimal for linear systems, computes a constant feedback gain matrix offline
- **MPC** (Model Predictive Control) — optimizes over a receding horizon, can handle constraints (speed limits, obstacle avoidance) explicitly
- **RL** (Reinforcement Learning) — learns a control policy from experience, flexible but data-hungry and hard to certify
- **BEV fusion** — bird's-eye view representations from multiple cameras for end-to-end perception-to-planning
- **End-to-end driving** — skip the modular pipeline entirely and learn perception + planning + control as a single neural network

### References

- Coulter, R. C. (1992) — *Implementation of the Pure Pursuit Path Tracking Algorithm*, CMU Robotics Institute Technical Report CMU-RI-TR-92-01
- Hoffmann, G. M., Tomlin, C. J., Montemerlo, M., & Thrun, S. (2007) — *Autonomous Automobile Trajectory Tracking for Off-Road Driving*, American Control Conference
- Hart, P. E., Nilsson, N. J., & Raphael, B. (1968) — *A Formal Basis for the Heuristic Determination of Minimum Cost Paths*, IEEE Transactions (the A* planner whose output we consume here)
- Rajamani, R. (2012) — *Vehicle Dynamics and Control*, Springer (comprehensive reference for bicycle models and steering controllers)